# Nearest-pattern rule versus RP-CNN on recall

Rules fixed before running:
- **Q1.** On the Step 2 recall settings (exact copies), the task is solved by a fixed lookup if the nearest-pattern rule reaches R² ≥ 0.99.
- **Q2.** With a noisy query, RP-CNN adds something beyond hand-written retrieval if its mean R² beats the better of the two rules by ≥ 0.02.

In [ ]:
from pathlib import Path
import sys
sys.path.insert(0, str(Path('../../src').resolve()))
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from IPython.display import display
from rpjepa.train import train_supervised
from rpjepa.experiment import SMOKE, get_split, cached_run, summarise

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
M, TAU, DEPTH, BATCH, LR = 3, 1, 2, 128, 1e-3
SIZES = {'train': 800, 'val': 200, 'test': 300} if SMOKE else {'train': 20000, 'val': 2000, 'test': 5000}
SEEDS = (0,) if SMOKE else (0, 1, 2)
EPOCHS, PATIENCE = (2, 1) if SMOKE else (40, 10)
D_RUN = 16 if SMOKE else 32
N_MAIN = 32 if SMOKE else 64
CACHE_ROOT = Path('../../data/processed/jepa/step2')

from numpy.lib.stride_tricks import sliding_window_view
TABLE_ROOT = Path('../../results/tables/jepa/step5') / ('smoke' if SMOKE else '')
TABLE_ROOT.mkdir(parents=True, exist_ok=True)
SPECS = {'rp_cnn': ('rp_cnn', {'rp_kind': 'D+series'}), 'raw_attn': ('raw_attn', {})}
NOISES = (0.5,) if SMOKE else (0.25, 0.5, 1.0)
GRID = ([(N_MAIN, 4, 0.0), (N_MAIN, 4, NOISES[0])] if SMOKE else
        [(n, 4, 0.0) for n in (32, 64, 128)] + [(64, p, 0.0) for p in (1, 2, 8)] + [(64, 4, q) for q in NOISES])
K = M
print({'device': DEVICE, 'smoke': SMOKE, 'grid': GRID})

def split(name, n_states, seed, n_pairs, noise):
    difficulty = {'n_pairs': int(n_pairs)} | ({'query_noise': float(noise)} if noise else {})
    return get_split('recall', name, SIZES[name], n_states, M, TAU, base_seed=seed, **difficulty)

## Rules

In [ ]:
def candidates(x):
    L = x.shape[1]
    q = x[:, L-K:]
    w = sliding_window_view(x[:, :L-K-1], K, axis=1)
    return ((w - q[:, None, :])**2).sum(-1), x[:, K:L-K]

def rule_nearest(x):
    d, nxt = candidates(x)
    return nxt[np.arange(len(x)), d.argmin(1)]

def rule_soft(x, h):
    d, nxt = candidates(x)
    z = -d / (2*h*h)
    w = np.exp(z - z.max(1, keepdims=True))
    return (w*nxt).sum(1) / w.sum(1)

def r2(y, p):
    return float(1 - ((y-p)**2).sum() / ((y-y.mean())**2).sum())

def fit_r2(p_tr, y_tr, p_te, y_te):
    b, a = np.polyfit(p_tr, y_tr, 1)
    return r2(y_te, a + b*p_te)

H_GRID = (0.05, 0.1, 0.2, 0.3, 0.5, 0.75, 1.0, 1.5, 2.0)

def rule_rows(n_states, n_pairs, noise, seed):
    tr, va, te = (split(s, n_states, seed, n_pairs, noise) for s in ('train', 'val', 'test'))
    base = {'N': n_states, 'n_pairs': n_pairs, 'query_noise': noise, 'seed': seed}
    hard = fit_r2(rule_nearest(tr.x), tr.y, rule_nearest(te.x), te.y)
    val = {h: fit_r2(rule_soft(tr.x, h), tr.y, rule_soft(va.x, h), va.y) for h in H_GRID}
    h = max(val, key=val.get)
    soft = fit_r2(rule_soft(tr.x, h), tr.y, rule_soft(te.x, h), te.y)
    return [base | {'method': 'rule_nearest', 'metric': hard}, base | {'method': 'rule_soft', 'h': h, 'metric': soft}]

rule_df = pd.DataFrame([r for g in GRID for s in SEEDS for r in rule_rows(*g, s)])
display(summarise(rule_df, ['method', 'N', 'n_pairs', 'query_noise'], 'metric').round(4))
display(rule_df[rule_df.method == 'rule_soft'].groupby('query_noise').h.agg(list))

## Trained models

In [ ]:
def train_row(spec, n_states, n_pairs, noise, seed):
    enc_name, enc_kw = SPECS[spec]
    tr, va, te = (split(s, n_states, seed, n_pairs, noise) for s in ('train', 'val', 'test'))
    key = {'spec_label': spec, 'enc_name': enc_name, 'enc_kw': enc_kw, 'd': D_RUN, 'depth': DEPTH, 'N': int(n_states),
           'n_pairs': int(n_pairs), 'task': 'recall', 'seed': int(seed), 'epochs': EPOCHS, 'patience': PATIENCE,
           'sizes': SIZES, 'batch': BATCH, 'lr': LR, 'm': M, 'tau': TAU}
    if noise:
        key['query_noise'] = float(noise)
    res = cached_run(CACHE_ROOT, key, lambda: train_supervised(
        enc_name, D_RUN, DEPTH, tr, va, te, n_states=n_states, m=M, tau=TAU, epochs=EPOCHS, batch=BATCH,
        lr=LR, seed=seed, patience=PATIENCE, device=DEVICE, enc_kw=enc_kw))
    print(spec, n_states, n_pairs, noise, seed, round(res['metric'], 4), f"{res['train_s']:.0f}s", flush=True)
    return {'method': spec, 'N': n_states, 'n_pairs': n_pairs, 'query_noise': noise, 'seed': seed,
            'metric': res['metric'], 'train_s': res['train_s'], 'flops': res['flops']}

model_df = pd.DataFrame([train_row(spec, *g, s) for g in GRID for spec in SPECS for s in SEEDS])

## Results

In [ ]:
all_df = pd.concat([rule_df, model_df], ignore_index=True)
all_df.to_csv(TABLE_ROOT / 'rule_vs_models.csv', index=False)
summary = summarise(all_df, ['method', 'N', 'n_pairs', 'query_noise'], 'metric')
table = summary.pivot_table(index=['N', 'n_pairs', 'query_noise'], columns='method', values='mean')
display(table.round(4))
summary.to_csv(TABLE_ROOT / 'rule_vs_models_summary.csv', index=False)

fig, ax = plt.subplots(figsize=(7, 4))
noisy = summary[(summary.N == N_MAIN) & (summary.n_pairs == 4)]
for method, g in noisy.groupby('method'):
    g = g.sort_values('query_noise')
    ax.errorbar(g.query_noise, g['mean'], yerr=g['sd'].fillna(0), marker='o', capsize=3, label=method)
ax.set(xlabel='Query noise (SD, raw units)', ylabel='Test R²', title=f'Recall, N={N_MAIN}, 4 pairs')
ax.grid(alpha=.25); ax.legend()
plt.tight_layout(); plt.show()

## Decision

In [ ]:
rows = []
for (n, p, q), g in table.iterrows():
    best_rule = max(g['rule_nearest'], g['rule_soft'])
    rows.append({'N': n, 'n_pairs': p, 'query_noise': q, 'rule_nearest': g['rule_nearest'], 'rule_soft': g['rule_soft'],
                 'rp_cnn': g['rp_cnn'], 'raw_attn': g['raw_attn'], 'rp_cnn_minus_best_rule': g['rp_cnn'] - best_rule,
                 'Q1_rule_solves': (g['rule_nearest'] >= .99) if q == 0 else None,
                 'Q2_cnn_beyond_rule': (g['rp_cnn'] - best_rule >= .02) if q > 0 else None})
decision = pd.DataFrame(rows)
display(decision.round(4))
decision.to_csv(TABLE_ROOT / 'decision.csv', index=False)
L = N_MAIN + (M - 1) * TAU
print('rule cost per window ≈', (L - 2*K) * K * 3, 'FLOPs; rp_cnn ≈', int(model_df[model_df.method == 'rp_cnn'].flops.iloc[0]))

## Findings

Full run: 3 seeds; train/val/test 20,000/2,000/5,000; RP-CNN and attention at d = 32, depth 2, 40 epochs, patience 10. Noise-free model rows come from the Step 2 cache; the noisy-query rows were trained here. Test R², mean of 3 seeds.

| setting | nearest rule | soft rule | RP-CNN | attention |
|---|---|---|---|---|
| N=32, 4 pairs | 1.000 | 0.999 | 0.999 | 0.909 |
| N=64, 1 / 2 / 4 / 8 pairs | 0.999–1.000 | 0.998–0.999 | 0.997–0.999 | 0.924 → 0.521 |
| N=128, 4 pairs | 1.000 | 0.998 | 0.997 | 0.529 |
| N=64, query noise 0.25 | 0.514 | 0.605 | 0.611 | 0.272 |
| query noise 0.5 | 0.130 | 0.240 | 0.228 | 0.087 |
| query noise 1.0 | 0.020 | 0.068 | 0.003 | 0.020 |

1. **Q1 holds: a fixed lookup solves the Step 2 recall task.** The nearest-pattern rule reaches R² ≥ 0.999 in every noise-free setting, at about 540 FLOPs per window against 287 M for RP-CNN at N = 64. RP-CNN learned this rule; it does not go beyond it.
2. **Q2 does not hold: with a noisy query, RP-CNN does not beat the best rule.** Its gap to the better rule is +0.006, −0.012 and −0.065 at noise 0.25, 0.5 and 1.0.
3. **But RP-CNN learned the soft rule, not the hard one.** At noise 0.25 and 0.5 it beats nearest-match (0.611 vs 0.514; 0.228 vs 0.130) and sits on the soft, distance-weighted rule. It has learned to average over near matches by closeness, without being told the noise level.
4. At noise 1.0 every method is near zero, and only the soft rule keeps a little signal (0.068). Raw attention is below the hard rule at every noise level.

Step 1 already reported the noise-free nearest-key oracle (0.999, ≈ 0.4 at query noise 0.3); new here are the trained models and the soft rule on the same noisy splits.

Caveats: the soft-rule bandwidth is chosen on validation windows; the rules are given the key length (3) and the lag (1), which the networks must learn; the noisy RP-CNN runs shared the GPU with another notebook, so their train_s is not comparable with Step 2.